In [4]:
import os

os.getcwd()

'/Users/waheed/Desktop/Project1'

In [5]:
os.listdir()

['.DS_Store', '.ipynb_checkpoints', 'data', 'notebooks']

In [6]:
os.listdir("data")

['data_dictionary_trip_records_hvfhs.pdf',
 'taxi_zone_lookup.csv',
 'fhvhv_tripdata_2025-01.parquet']

In [8]:
%pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 20.3 MB/s  0:00:003.2 MB/s eta 0:00:0101

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: /Users/waheed/miniconda3/envs/ml-main/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import duckdb

duckdb.__version__

'1.5.6'

In [11]:
import duckdb

con = duckdb.connect()

df_preview = con.execute("""
    SELECT *
    FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
    LIMIT 5
""").df()

df_preview

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag,cbd_congestion_fee
0,HV0003,B03404,B03404,2025-01-01 00:28:07,2025-01-01 00:31:17,2025-01-01 00:33:25,2025-01-01 00:54:24,148,211,1.320,...,2.75,0.0,0.0,15.71,N,N,N,N,N,0.0
1,HV0005,B03406,None,2025-01-01 00:18:33,NaT,2025-01-01 00:29:49,2025-01-01 00:41:52,127,244,3.313,...,0.00,0.0,0.0,23.57,N,N,N,N,N,0.0
2,HV0003,B03404,B03404,2025-01-01 00:28:22,2025-01-01 00:31:52,2025-01-01 00:32:39,2025-01-01 01:20:33,132,181,13.430,...,0.00,2.5,0.0,47.75,N,N,N,N,N,0.0
3,HV0003,B03404,B03404,2025-01-01 00:27:13,2025-01-01 00:33:58,2025-01-01 00:34:55,2025-01-01 00:39:19,76,76,0.820,...,0.00,0.0,0.0,12.20,N,N,N,N,N,0.0
4,HV0003,B03404,B03404,2025-01-01 00:33:29,2025-01-01 00:45:46,2025-01-01 00:46:19,2025-01-01 00:53:56,76,76,1.610,...,0.00,0.0,0.0,17.67,N,N,N,N,N,0.0


In [12]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,hvfhs_license_num,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,originating_base_num,VARCHAR,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,pickup_datetime,TIMESTAMP,YES,None,None,None
6,dropoff_datetime,TIMESTAMP,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,trip_miles,DOUBLE,YES,None,None,None


In [13]:
profile = con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT hvfhs_license_num) AS unique_providers,

    MIN(request_datetime) AS earliest_request,
    MAX(request_datetime) AS latest_request,

    MIN(trip_miles) AS min_trip_miles,
    MAX(trip_miles) AS max_trip_miles,

    MIN(trip_time) AS min_trip_time,
    MAX(trip_time) AS max_trip_time,

    MIN(base_passenger_fare) AS min_fare,
    MAX(base_passenger_fare) AS max_fare,

    MIN(driver_pay) AS min_driver_pay,
    MAX(driver_pay) AS max_driver_pay

FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

profile

,total_rows,unique_providers,earliest_request,latest_request,min_trip_miles,max_trip_miles,min_trip_time,max_trip_time,min_fare,max_fare,min_driver_pay,max_driver_pay
0,20405666,2,2024-12-31 22:31:09,2025-02-01 00:10:00,0.0,507.22,0,29402,-27.98,1420.4,-55.02,994.2


In [14]:
con.execute("""
SELECT
    hvfhs_license_num,
    COUNT(*) AS trips
FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
GROUP BY hvfhs_license_num
ORDER BY trips DESC
""").df()

,hvfhs_license_num,trips
0,HV0003,15356455
1,HV0005,5049211


In [15]:
con.execute("""
SELECT
    shared_request_flag,
    shared_match_flag,
    wav_request_flag,
    wav_match_flag,
    COUNT(*) AS rows
FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
GROUP BY ALL
ORDER BY rows DESC
LIMIT 20
""").df()

,shared_request_flag,shared_match_flag,wav_request_flag,wav_match_flag,rows
0,N,N,N,N,17779200
1,N,N,N,Y,1737620
2,Y,Y,N,N,436405
3,Y,N,N,N,335176
4,N,N,Y,Y,51150
5,Y,Y,N,Y,35525
6,Y,N,N,Y,30166
7,N,Y,N,N,372
8,N,Y,N,Y,52


In [16]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(request_datetime) AS earliest_request,
    MAX(request_datetime) AS latest_request,

    MIN(trip_miles) AS min_trip_miles,
    MAX(trip_miles) AS max_trip_miles,

    MIN(trip_time) AS min_trip_time,
    MAX(trip_time) AS max_trip_time,

    MIN(base_passenger_fare) AS min_fare,
    MAX(base_passenger_fare) AS max_fare,

    MIN(driver_pay) AS min_driver_pay,
    MAX(driver_pay) AS max_driver_pay

FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

,total_rows,earliest_request,latest_request,min_trip_miles,max_trip_miles,min_trip_time,max_trip_time,min_fare,max_fare,min_driver_pay,max_driver_pay
0,20405666,2024-12-31 22:31:09,2025-02-01 00:10:00,0.0,507.22,0,29402,-27.98,1420.4,-55.02,994.2


In [17]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) - COUNT(request_datetime) AS missing_request_datetime,
    COUNT(*) - COUNT(on_scene_datetime) AS missing_on_scene_datetime,
    COUNT(*) - COUNT(pickup_datetime) AS missing_pickup_datetime,
    COUNT(*) - COUNT(dropoff_datetime) AS missing_dropoff_datetime,

    COUNT(*) - COUNT(PULocationID) AS missing_PU_location,
    COUNT(*) - COUNT(DOLocationID) AS missing_DO_location,

    COUNT(*) - COUNT(trip_miles) AS missing_trip_miles,
    COUNT(*) - COUNT(trip_time) AS missing_trip_time,

    COUNT(*) - COUNT(base_passenger_fare) AS missing_base_fare,
    COUNT(*) - COUNT(driver_pay) AS missing_driver_pay

FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

,total_rows,missing_request_datetime,missing_on_scene_datetime,missing_pickup_datetime,missing_dropoff_datetime,missing_PU_location,missing_DO_location,missing_trip_miles,missing_trip_time,missing_base_fare,missing_driver_pay
0,20405666,0,5035135,0,0,0,0,0,0,0,0


In [18]:
con.execute("""
SELECT
    MIN(PULocationID) AS min_PU,
    MAX(PULocationID) AS max_PU,
    COUNT(DISTINCT PULocationID) AS unique_PU_zones,

    MIN(DOLocationID) AS min_DO,
    MAX(DOLocationID) AS max_DO,
    COUNT(DISTINCT DOLocationID) AS unique_DO_zones

FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

,min_PU,max_PU,unique_PU_zones,min_DO,max_DO,unique_DO_zones
0,1,265,262,1,265,261


In [19]:
import pandas as pd

zones = pd.read_csv("data/taxi_zone_lookup.csv")

zones.shape

(265, 4)

In [20]:
zones.head(10)

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [21]:
zones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   LocationID    265 non-null    int64 
 1   Borough       264 non-null    object
 2   Zone          264 non-null    object
 3   service_zone  263 non-null    object
dtypes: int64(1), object(3)
memory usage: 8.4+ KB


In [22]:
zones["Borough"].value_counts(dropna=False)

Borough
Queens           69
Manhattan        69
Brooklyn         61
Bronx            43
Staten Island    20
EWR               1
Unknown           1
NaN               1
Name: count, dtype: int64

In [23]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(request_datetime) AS earliest_request,
    MAX(request_datetime) AS latest_request,

    MIN(trip_miles) AS min_trip_miles,
    MAX(trip_miles) AS max_trip_miles,

    MIN(trip_time) AS min_trip_time,
    MAX(trip_time) AS max_trip_time,

    MIN(base_passenger_fare) AS min_fare,
    MAX(base_passenger_fare) AS max_fare,

    MIN(driver_pay) AS min_driver_pay,
    MAX(driver_pay) AS max_driver_pay

FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

,total_rows,earliest_request,latest_request,min_trip_miles,max_trip_miles,min_trip_time,max_trip_time,min_fare,max_fare,min_driver_pay,max_driver_pay
0,20405666,2024-12-31 22:31:09,2025-02-01 00:10:00,0.0,507.22,0,29402,-27.98,1420.4,-55.02,994.2


In [24]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) - COUNT(request_datetime) AS missing_request_datetime,
    COUNT(*) - COUNT(on_scene_datetime) AS missing_on_scene_datetime,
    COUNT(*) - COUNT(pickup_datetime) AS missing_pickup_datetime,
    COUNT(*) - COUNT(dropoff_datetime) AS missing_dropoff_datetime,

    COUNT(*) - COUNT(PULocationID) AS missing_PU_location,
    COUNT(*) - COUNT(DOLocationID) AS missing_DO_location,

    COUNT(*) - COUNT(trip_miles) AS missing_trip_miles,
    COUNT(*) - COUNT(trip_time) AS missing_trip_time,

    COUNT(*) - COUNT(base_passenger_fare) AS missing_base_fare,
    COUNT(*) - COUNT(driver_pay) AS missing_driver_pay

FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

,total_rows,missing_request_datetime,missing_on_scene_datetime,missing_pickup_datetime,missing_dropoff_datetime,missing_PU_location,missing_DO_location,missing_trip_miles,missing_trip_time,missing_base_fare,missing_driver_pay
0,20405666,0,5035135,0,0,0,0,0,0,0,0


In [25]:
con.execute("""
SELECT
    MIN(PULocationID) AS min_PU,
    MAX(PULocationID) AS max_PU,
    COUNT(DISTINCT PULocationID) AS unique_PU_zones,

    MIN(DOLocationID) AS min_DO,
    MAX(DOLocationID) AS max_DO,
    COUNT(DISTINCT DOLocationID) AS unique_DO_zones

FROM read_parquet('data/fhvhv_tripdata_2025-01.parquet')
""").df()

,min_PU,max_PU,unique_PU_zones,min_DO,max_DO,unique_DO_zones
0,1,265,262,1,265,261


In [33]:
from pathlib import Path

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    
    queries = {}
    current_name = None
    current_sql = []

    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()

            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)

    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()

    return queries

In [34]:
setup = load_queries("sql/00_setup.sql")
for name in setup:
    con.execute(setup[name])

con.execute("SELECT COUNT(*) AS n_rows FROM trips_raw").df()

,n_rows
0,20405666


In [36]:
def run(queries, name):
    return con.execute(queries[name]).df()

probe = load_queries("sql/00_scope_probe.sql")
display(run(probe, "ts_ranges").T)
display(run(probe, "month_crosstab"))
display(run(probe, "onscene_by_company"))
display(run(probe, "onscene_by_wav"))

,0
min_request,2024-12-31 22:31:09
max_request,2025-02-01 00:10:00
min_on_scene,2024-12-31 23:25:35
max_on_scene,2025-01-31 23:59:59
min_pickup,2025-01-01 00:00:00
max_pickup,2025-01-31 23:59:59
min_dropoff,2025-01-01 00:01:48
max_dropoff,2025-02-01 01:50:34
n_pickup_before_request,199788


,request_month,pickup_month,dropoff_month,n_rows
0,2025-01-01,2025-01-01,2025-01-01,20393027
1,2025-01-01,2025-01-01,2025-02-01,11470
2,2024-12-01,2025-01-01,2025-01-01,1164
3,2025-02-01,2025-01-01,2025-02-01,5


,hvfhs_license_num,n_rows,n_with_on_scene,pct_with_on_scene
0,HV0003,15356455,15356455,100.00
1,HV0005,5049211,14076,0.28


,wav_match_flag,n_rows,n_with_on_scene,pct_with_on_scene
0,N,18551153,14165575,76.36
1,Y,1854513,1204956,64.97
